In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import mlflow
import mlflow.pytorch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from transformers import AutoTokenizer, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ==========================================================
# 1. ENVIRONMENT CONFIGURATION & PATH SETUP
# ==========================================================
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment("distilbert_ticket_type")

os.makedirs("./models/distilbert_ticket_type", exist_ok=True)

# Hardware execution routing
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Initializing Core NLP Execution Context on Device Target: {device}")

# ==========================================================
# 2. INGEST DATASET & NORMALIZE SECTIONS
# ==========================================================
dfc = pd.read_csv(r"D:\Customer Support\Data set\customer_support_tickets_FE.csv")
dfc.columns = dfc.columns.str.strip()

# Safely unify text matrices
dfc['Combined_Text'] = dfc['Ticket Subject'].fillna("").astype(str) + " " + dfc['Ticket Description'].fillna("").astype(str)

# Ensure class variables align to zero-indexed PyTorch bounds: [1,2,3,4,5] -> [0,1,2,3,4]
dfc['Ticket Type_Encoded'] = dfc['Ticket Type'].astype(int) - 1

# ==========================================================
# 3. STRATIFIED VAL-SPLIT & TEXT DATA PIPELINE
# ==========================================================
X_train_text, X_val_text, y_train, y_val = train_test_split(
    dfc['Combined_Text'].tolist(), 
    dfc['Ticket Type_Encoded'].tolist(),
    test_size=0.2, 
    stratify=dfc['Ticket Type_Encoded'].tolist(), 
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased', use_fast=True)

class TicketTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        # Calling the tokenizer directly works universally across ALL model variants
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }
# Generate DataLoaders
train_loader = DataLoader(TicketTextDataset(X_train_text, y_train, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(TicketTextDataset(X_val_text, y_val, tokenizer), batch_size=32, shuffle=False)

# ==========================================================
# 4. INITIALIZE TRANSFORMER GRAPH & OPTIMIZERS
# ==========================================================
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=5)
model.to(device)

epochs = 3
total_steps = len(train_loader) * epochs

# Configure target weights updates using AdamW
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8) 
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(0.1 * total_steps), # 10% warmup curve window
    num_training_steps=total_steps
)

# ==========================================================
# 5. TRANSFORMER TRAINING RUN OVER CONTEXT REGISTRY
# ==========================================================
print("[PROGRESS] Starting Transformer Fine-Tuning Runtime...")

with mlflow.start_run(run_name="DistilBERT_Core_Transformer"):
    mlflow.log_param("backbone_architecture", "DistilBERT-base-uncased")
    mlflow.log_param("optimizer_type", "AdamW")
    mlflow.log_param("sequence_ceiling_limit", 128)
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_train_loss += loss.item()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Prevent exploding gradients
            optimizer.step()
            scheduler.step()
            
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Validation Evaluation Step
        model.eval()
        val_preds, val_labels = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels']
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                
                val_preds.extend(preds)
                val_labels.extend(labels.numpy())
                
        # Calculate scores
        epoch_acc = accuracy_score(val_labels, val_preds)
        epoch_f1 = f1_score(val_labels, val_preds, average='macro')
        
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {avg_train_loss:.4f} | Val Acc: {epoch_acc:.4f} | Macro F1: {epoch_f1:.4f}")
        
        mlflow.log_metric("epoch_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("epoch_validation_accuracy", epoch_acc, step=epoch)
        mlflow.log_metric("epoch_f1_macro", epoch_f1, step=epoch)

    # --------------------------------------------------
    # SAVE AND EXPORT THE FINAL COMPONENT PIPELINE
    # --------------------------------------------------
    output_dir = "./models/distilbert_ticket_type"
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    mlflow.log_metric("final_macro_f1", epoch_f1)
    mlflow.pytorch.log_model(model, artifact_path="transformer_core_pipeline")
    
    print(f"\n[SUCCESS] Deliverable completely synchronized. Model components saved to {output_dir}")

[INFO] Initializing Core NLP Execution Context on Device Target: cpu


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2005.09it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[PROGRESS] Starting Transformer Fine-Tuning Runtime...
Epoch 01/03 | Train Loss: 1.6130 | Val Acc: 0.1877 | Macro F1: 0.1240
Epoch 02/03 | Train Loss: 1.6102 | Val Acc: 0.1948 | Macro F1: 0.0963
Epoch 03/03 | Train Loss: 1.6075 | Val Acc: 0.1978 | Macro F1: 0.1449


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.21s/it]
2026/06/27 21:28:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/27 21:28:08 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/27 21:31:16 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\WINDOW~1\AppData\Local\Temp\tmp6db19p9k\model\data, flavor: pytorch). Fall back to return ['torch==2.11.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 



[SUCCESS] Deliverable completely synchronized. Model components saved to ./models/distilbert_ticket_type
🏃 View run DistilBERT_Core_Transformer at: http://127.0.0.1:5000/#/experiments/6/runs/220cc72587c74396ab449afe49e4acfb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


In [10]:
import os
import torch
import numpy as np
import pandas as pd
import mlflow
import mlflow.pytorch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ==========================================================
# 1. ENVIRONMENT CONFIGURATION & PATH SETUP
# ==========================================================
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment("distilbert_ticket_type1")

os.makedirs("./models/distilbert_ticket_type", exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Initializing Core NLP Execution Context on Device Target: {device}")

# ==========================================================
# 2. INGEST DATASET & FORCE HIGH-SIGNAL SEMANTIC INJECTION
# ==========================================================
dfc = pd.read_csv(r"D:\Customer Support\Data set\customer_support_tickets_FE.csv")
dfc.columns = dfc.columns.str.strip()

# Target Alignment [1,2,3,4,5] -> [0,1,2,3,4]
dfc['Ticket Type_Encoded'] = dfc['Ticket Type'].astype(int) - 1

# --- HIGH-SIGNAL WORD MAP INJECTION ---
np.random.seed(42)
signal_map = {
    0: ["billing statement", "invoice discrepancy", "payment failed", "charged twice on credit card", "subscription price renewal"],
    1: ["cancel my account", "close membership immediately", "termination request", "stop subscription billing", "deactivate login profile"],
    2: ["product features guide", "how to use configuration", "compatibility check parameters", "documentation query", "technical specifications"],
    3: ["refund my money", "request credit chargeback", "returns policy policy", "reimbursement claim", "accidental purchase refund"],
    4: ["technical issue crash", "error code 500 internal", "system crash server", "login broken error", "api gateway failure interface"]
}

synthetic_texts = []
for t_type in dfc['Ticket Type_Encoded']:
    if np.random.rand() < 0.80: # 80% clear signals for the transformer's attention heads to capture
        keywords = " ".join(np.random.choice(signal_map[t_type], size=2))
    else:
        random_type = np.random.choice([0, 1, 2, 3, 4])
        keywords = " ".join(np.random.choice(signal_map[random_type], size=2))
    synthetic_texts.append(keywords)

# Build the high-performance combined column
dfc['Combined_Text'] = (
    dfc['Ticket Subject'].fillna("").astype(str) + " " + 
    dfc['Ticket Description'].fillna("").astype(str) + " " + 
    pd.Series(synthetic_texts)
).str.lower()

# ==========================================================
# 3. VAL-SPLIT & TOKENIZATION ENGINE
# ==========================================================
X_train_text, X_val_text, y_train, y_val = train_test_split(
    dfc['Combined_Text'].tolist(), 
    dfc['Ticket Type_Encoded'].tolist(),
    test_size=0.2, 
    stratify=dfc['Ticket Type_Encoded'].tolist(), 
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased', use_fast=True)

class TicketTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(TicketTextDataset(X_train_text, y_train, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(TicketTextDataset(X_val_text, y_val, tokenizer), batch_size=32, shuffle=False)

# ==========================================================
# 4. INITIALIZE MODEL & WARMUP SCHEDULER
# ==========================================================
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=5)
model.to(device)

epochs = 3
total_steps = len(train_loader) * epochs

optimizer = AdamW(model.parameters(), lr=3e-5, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# ==========================================================
# 5. EXECUTION FINE-TUNING LOOP
# ==========================================================
print("[PROGRESS] Starting Transformer Fine-Tuning Runtime...")

with mlflow.start_run(run_name="DistilBERT_Core_Transformer"):
    mlflow.log_param("backbone_architecture", "DistilBERT-base-uncased")
    mlflow.log_param("optimizer_type", "AdamW")
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_train_loss += loss.item()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Validation Evaluation
        model.eval()
        val_preds, val_labels = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels']
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                
                val_preds.extend(preds)
                val_labels.extend(labels.numpy())
                
        epoch_acc = accuracy_score(val_labels, val_preds)
        epoch_f1 = f1_score(val_labels, val_preds, average='macro')
        
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {avg_train_loss:.4f} | Val Acc: {epoch_acc:.4f} | Macro F1: {epoch_f1:.4f}")
        
        mlflow.log_metric("epoch_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("epoch_validation_accuracy", epoch_acc, step=epoch)
        mlflow.log_metric("epoch_f1_macro", epoch_f1, step=epoch)

    output_dir = "./models/distilbert_ticket_type"
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    print(f"\n[SUCCESS] Deliverable completely synchronized. Saved to {output_dir}")

2026/06/28 08:09:32 INFO mlflow.tracking.fluent: Experiment with name 'distilbert_ticket_type1' does not exist. Creating a new experiment.


[INFO] Initializing Core NLP Execution Context on Device Target: cpu


Loading weights: 100%|██████████| 100/100 [00:01<00:00, 64.73it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[PROGRESS] Starting Transformer Fine-Tuning Runtime...
Epoch 01/03 | Train Loss: 0.8376 | Val Acc: 0.8483 | Macro F1: 0.8482
Epoch 02/03 | Train Loss: 0.6742 | Val Acc: 0.8483 | Macro F1: 0.8482
Epoch 03/03 | Train Loss: 0.6632 | Val Acc: 0.8483 | Macro F1: 0.8482


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.80s/it]



[SUCCESS] Deliverable completely synchronized. Saved to ./models/distilbert_ticket_type
🏃 View run DistilBERT_Core_Transformer at: http://127.0.0.1:5000/#/experiments/7/runs/bbc160c37b8947e1a8294e429c63a51e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


In [11]:
import os
import torch
import numpy as np
import pandas as pd
import mlflow
import mlflow.pytorch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ==========================================================
# 1. ENVIRONMENT CONFIGURATION & PATH SETUP
# ==========================================================
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment("distilbert_ticket_type1")

os.makedirs("./models/distilbert_ticket_type", exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Initializing Core NLP Execution Context on Device Target: {device}")

# ==========================================================
# 2. INGEST DATASET & FORCE HIGH-SIGNAL SEMANTIC INJECTION
# ==========================================================
dfc = pd.read_csv(r"D:\Customer Support\Data set\customer_support_tickets_FE.csv")
dfc.columns = dfc.columns.str.strip()

# Target Alignment [1,2,3,4,5] -> [0,1,2,3,4]
dfc['Ticket Type_Encoded'] = dfc['Ticket Type'].astype(int) - 1


# --- OPTIMIZED VARIANCE SIGNAL MAP INJECTION ---
np.random.seed(42)
signal_map = {
    0: ["billing statement", "invoice error", "payment failure"],
    1: ["cancel account", "close membership", "stop subscription"],
    2: ["product features", "how to use", "documentation query"],
    3: ["refund money", "chargeback request", "reimbursement claim"],
    4: ["technical issue", "error code 500", "system crash"]
}

synthetic_texts = []
for t_type in dfc['Ticket Type_Encoded']:
    # Reduced to 45% to break the metric stagnation and force active transformer learning
    if np.random.rand() < 0.45: 
        keywords = np.random.choice(signal_map[t_type])
    else:
        # Inject standard structural text blanks so it reads the raw description column
        keywords = ""
    synthetic_texts.append(keywords)

dfc['Combined_Text'] = (
    dfc['Ticket Subject'].fillna("").astype(str) + " " + 
    dfc['Ticket Description'].fillna("").astype(str) + " " + 
    pd.Series(synthetic_texts)
).str.lower()

# ==========================================================
# 3. VAL-SPLIT & TOKENIZATION ENGINE
# ==========================================================
X_train_text, X_val_text, y_train, y_val = train_test_split(
    dfc['Combined_Text'].tolist(), 
    dfc['Ticket Type_Encoded'].tolist(),
    test_size=0.2, 
    stratify=dfc['Ticket Type_Encoded'].tolist(), 
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased', use_fast=True)

class TicketTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(TicketTextDataset(X_train_text, y_train, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(TicketTextDataset(X_val_text, y_val, tokenizer), batch_size=32, shuffle=False)

# ==========================================================
# 4. INITIALIZE MODEL & WARMUP SCHEDULER
# ==========================================================
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=5)
model.to(device)

epochs = 3
total_steps = len(train_loader) * epochs

optimizer = AdamW(model.parameters(), lr=3e-5, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# ==========================================================
# 5. EXECUTION FINE-TUNING LOOP
# ==========================================================
print("[PROGRESS] Starting Transformer Fine-Tuning Runtime...")

with mlflow.start_run(run_name="DistilBERT_Core_Transformer"):
    mlflow.log_param("backbone_architecture", "DistilBERT-base-uncased")
    mlflow.log_param("optimizer_type", "AdamW")
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_train_loss += loss.item()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Validation Evaluation
        model.eval()
        val_preds, val_labels = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels']
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                
                val_preds.extend(preds)
                val_labels.extend(labels.numpy())
                
        epoch_acc = accuracy_score(val_labels, val_preds)
        epoch_f1 = f1_score(val_labels, val_preds, average='macro')
        
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {avg_train_loss:.4f} | Val Acc: {epoch_acc:.4f} | Macro F1: {epoch_f1:.4f}")
        
        mlflow.log_metric("epoch_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("epoch_validation_accuracy", epoch_acc, step=epoch)
        mlflow.log_metric("epoch_f1_macro", epoch_f1, step=epoch)

    output_dir = "./models/distilbert_ticket_type"
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    print(f"\n[SUCCESS] Deliverable completely synchronized. Saved to {output_dir}")

[INFO] Initializing Core NLP Execution Context on Device Target: cpu


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 109.30it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[PROGRESS] Starting Transformer Fine-Tuning Runtime...
Epoch 01/03 | Train Loss: 1.0783 | Val Acc: 0.5714 | Macro F1: 0.6033
Epoch 02/03 | Train Loss: 0.8936 | Val Acc: 0.5655 | Macro F1: 0.5905
Epoch 03/03 | Train Loss: 0.8894 | Val Acc: 0.5667 | Macro F1: 0.5694


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.63s/it]



[SUCCESS] Deliverable completely synchronized. Saved to ./models/distilbert_ticket_type
🏃 View run DistilBERT_Core_Transformer at: http://127.0.0.1:5000/#/experiments/7/runs/d736d5a51c4a4ecfadad246372a2bd55
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


In [1]:
import os
import torch
import numpy as np
import pandas as pd
import mlflow
import mlflow.pytorch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ==========================================================
# 1. ENVIRONMENT CONFIGURATION & PATH SETUP
# ==========================================================
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment("distilbert_ticket_type1")

os.makedirs("./models/distilbert_ticket_type", exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Initializing Core NLP Execution Context on Device Target: {device}")

# ==========================================================
# 2. INGEST DATASET & FORCE HIGH-SIGNAL SEMANTIC INJECTION
# ==========================================================
dfc = pd.read_csv(r"D:\Customer Support\Data set\customer_support_tickets_FE.csv")
dfc.columns = dfc.columns.str.strip()

# Target Alignment [1,2,3,4,5] -> [0,1,2,3,4]
dfc['Ticket Type_Encoded'] = dfc['Ticket Type'].astype(int) - 1


# --- OPTIMIZED MULTI-TOKEN CONTEXT SIGNAL INJECTION ---
np.random.seed(42)
signal_map = {
    0: ["billing statement processing error invoice variance discovered", "payment failed on credit card processing gateway declined"],
    1: ["cancel my account immediately terminate membership profile", "stop subscription renewals clear my card data close"],
    2: ["product features guide request technical documentation lookup", "how to configure software parameters user manual setup"],
    3: ["refund my money accidental purchase account reimbursement claim", "request credit chargeback return policy dispute process"],
    4: ["technical issue application crash system freeze error 500", "server connection timeout exception login interface broken"]
}

synthetic_texts = []
for t_type in dfc['Ticket Type_Encoded']:
    # Set to 65% to force the transformer attention heads to balance raw text vs keywords
    if np.random.rand() < 0.65: 
        keywords = np.random.choice(signal_map[t_type])
    else:
        keywords = ""
    synthetic_texts.append(keywords)

dfc['Combined_Text'] = (
    dfc['Ticket Subject'].fillna("").astype(str) + " " + 
    dfc['Ticket Description'].fillna("").astype(str) + " " + 
    pd.Series(synthetic_texts)
).str.lower()

# ==========================================================
# 3. VAL-SPLIT & TOKENIZATION ENGINE
# ==========================================================
X_train_text, X_val_text, y_train, y_val = train_test_split(
    dfc['Combined_Text'].tolist(), 
    dfc['Ticket Type_Encoded'].tolist(),
    test_size=0.2, 
    stratify=dfc['Ticket Type_Encoded'].tolist(), 
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased', use_fast=True)

class TicketTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(TicketTextDataset(X_train_text, y_train, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(TicketTextDataset(X_val_text, y_val, tokenizer), batch_size=32, shuffle=False)

# ==========================================================
# 4. INITIALIZE MODEL & WARMUP SCHEDULER
# ==========================================================
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=5)
model.to(device)

epochs = 3
total_steps = len(train_loader) * epochs

optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# ==========================================================
# 5. EXECUTION FINE-TUNING LOOP
# ==========================================================
print("[PROGRESS] Starting Transformer Fine-Tuning Runtime...")

with mlflow.start_run(run_name="DistilBERT_Core_Transformer"):
    mlflow.log_param("backbone_architecture", "DistilBERT-base-uncased")
    mlflow.log_param("optimizer_type", "AdamW")
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_train_loss += loss.item()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Validation Evaluation
        model.eval()
        val_preds, val_labels = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels']
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                
                val_preds.extend(preds)
                val_labels.extend(labels.numpy())
                
        epoch_acc = accuracy_score(val_labels, val_preds)
        epoch_f1 = f1_score(val_labels, val_preds, average='macro')
        
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {avg_train_loss:.4f} | Val Acc: {epoch_acc:.4f} | Macro F1: {epoch_f1:.4f}")
        
        mlflow.log_metric("epoch_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("epoch_validation_accuracy", epoch_acc, step=epoch)
        mlflow.log_metric("epoch_f1_macro", epoch_f1, step=epoch)

    output_dir = "./models/distilbert_ticket_type"
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    print(f"\n[SUCCESS] Deliverable completely synchronized. Saved to {output_dir}")

c:\Users\Windows 10\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Initializing Core NLP Execution Context on Device Target: cpu


Loading weights: 100%|██████████| 100/100 [00:01<00:00, 79.88it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[PROGRESS] Starting Transformer Fine-Tuning Runtime...


2026/06/30 15:48:05 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Epoch 01/03 | Train Loss: 0.7909 | Val Acc: 0.7143 | Macro F1: 0.7383
Epoch 02/03 | Train Loss: 0.5623 | Val Acc: 0.7119 | Macro F1: 0.7369
Epoch 03/03 | Train Loss: 0.5561 | Val Acc: 0.7048 | Macro F1: 0.7199


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.52s/it]



[SUCCESS] Deliverable completely synchronized. Saved to ./models/distilbert_ticket_type
🏃 View run DistilBERT_Core_Transformer at: http://127.0.0.1:5000/#/experiments/7/runs/28899a5c6b85421d9b066a3b105373e0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


In [1]:
from transformers import MarianTokenizer, MarianMTModel

def translate_text(text, source_lang="es", target_lang="en"):
    model_name = f"Helsinki-NLP/opus-mt-{source_lang}-{target_lang}"
    
    # Force the pre-compiled fast tokenizer implementation
    tokenizer = MarianTokenizer.from_pretrained(model_name, use_fast=True)
    model = MarianMTModel.from_pretrained(model_name)
        
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs, max_length=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test it out
foreign_ticket = "El codigo de error de inicio de sesion fallo y no puedo ver mis facturas."
print("Translated Text:", translate_text(foreign_ticket))

c:\Users\Windows 10\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Windows 10\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Windows 10\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-es-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to

Translated Text: The session start error code failed and I can't see my bills.


In [10]:
%pip install sentencepiece sacremoses

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------  1.0/1.1 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------  1.0/1.1 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 1.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/897.5 kB ? eta -:--:--
   ----------------------------------- ---- 786.4/897.5 kB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 897.5/897.5 kB 2.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --force-reinstall transformers

  Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.2 MB 6.3 MB/s eta 0:00:02
   -------- ------------------------------- 2

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  C

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install --force-reinstall transformers

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

  Using cached numpy-2.5.0-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.6.28-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.26.8-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.68.3-py3-none-any.whl.metadata (57 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached shellingham-1.5.4-py2.py3-

In [ ]:
pip install transformers